In [1]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import scipy

from qd_solve.spaces.finite_difference import *

In [2]:
x0, xf = -10, 10
num_steps = 1000
space = FiniteDifference(x0, xf, num_steps)
L = FiniteDifferenceLaplacian()
T = -0.5 * L
l1, l2 = T.spectral_bounds(space)
a = 0.5 * (l2 - 1)

num_iterations = 50
h = -1j * 0.01
Is = scipy.special.iv(jnp.arange(num_iterations), h * a)
Is

array([ 9.56387685e-02+0.00000000e+00j,  0.00000000e+00+1.25855138e-01j,
        1.05709194e-01+6.93889390e-18j, -1.38777878e-17+1.08938283e-01j,
        1.31859612e-01-1.38777878e-17j,  0.00000000e+00+6.67347668e-02j,
        1.58558858e-01+1.38777878e-17j,  8.67361738e-19-9.38870986e-03j,
        1.53300129e-01-4.40202175e-16j, -3.04767653e-16-1.07520419e-01j,
        7.58699415e-02-2.23630433e-16j, -5.00659862e-16-1.68228514e-01j,
       -7.22007646e-02+2.20605740e-16j, -3.25727189e-16-9.89019142e-02j,
       -1.75079331e-01+6.16696005e-16j,  3.37735642e-16+9.72261622e-02j,
       -5.83845975e-02+2.03375396e-16j,  5.74491951e-16+1.71973396e-01j,
        1.75546008e-01-4.91714630e-16j, -2.22281692e-16-8.08634222e-02j,
        5.26090187e-02-1.45927578e-16j, -3.40197897e-16-1.65054690e-01j,
       -2.24738331e-01+2.10236894e-16j,  0.00000000e+00+2.30563895e-01j,
        1.99584101e-01+2.48712592e-16j,  2.75049804e-16-1.52714234e-01j,
       -1.05905466e-01-2.06198738e-16j, -1.36375016

In [4]:
def miller(carry, idx):
    s_next, s = carry 
    s_prev = s_next + (2 * idx) / (h * a) * s
    return (s, s_prev), s_prev

idx_list = jnp.arange(1, num_iterations + 50)
_, Is_miller = jax.lax.scan(miller, (0j, 1j), idx_list[::-1])
Is_miller = Is_miller[::-1]

ks = jnp.arange(Is_miller.shape[0])
S  = jnp.sum(jnp.where(ks == 0, 1.0, 2.0) * Is_miller)   # Σ (2-δ_k0) Ĩ_k
Is_miller = Is_miller * jnp.exp(h * a) / S   
Is_miller[:num_iterations]

Array([ 9.56387685e-02+2.32259337e-17j, -2.95602792e-17+1.25855138e-01j,
        1.05709194e-01+2.74488307e-17j, -2.74488307e-17+1.08938283e-01j,
        1.31859612e-01+3.37831763e-17j, -1.68915881e-17+6.67347668e-02j,
        1.58558858e-01+3.80060733e-17j,  2.11144852e-18-9.38870986e-03j,
        1.53300129e-01+3.80060733e-17j,  2.74488307e-17-1.07520419e-01j,
        7.58699415e-02+1.90030367e-17j,  4.22289703e-17-1.68228514e-01j,
       -7.22007646e-02-1.68915881e-17j,  2.53373822e-17-9.89019142e-02j,
       -1.75079331e-01-4.22289703e-17j, -2.32259337e-17+9.72261622e-02j,
       -5.83845975e-02-1.47801396e-17j, -4.22289703e-17+1.71973396e-01j,
        1.75546008e-01+4.22289703e-17j,  1.90030367e-17-8.08634222e-02j,
        5.26090187e-02+1.37244154e-17j,  3.80060733e-17-1.65054690e-01j,
       -2.24738331e-01-5.48976614e-17j, -5.91205585e-17+2.30563895e-01j,
        1.99584101e-01+5.06747644e-17j,  3.80060733e-17-1.52714234e-01j,
       -1.05905466e-01-2.74488307e-17j, -1.68915881

In [ ]:
r_idx = 1

idx_list = jnp.arange(
    num_iterations - r_idx * num_mill_iterations, 
    num_iterations - (r_idx + 1) * num_mill_iterations, -1)

idx_list

In [ ]:
idx = 0

In [ ]:
jnp.where(idx == 0, 1.0, 2.0)